# Open-Meteo Data Exploration

This notebook downloads historical weather data for Eluru and Nellore and performs 6-hourly sampling for analysis.

In [5]:
import requests
import pandas as pd
import os

# Regions and coordinates
regions = {
    "Eluru": {"lat": 16.7119, "lon": 81.0949},
    "Nellore": {"lat": 14.4426, "lon": 79.9865}
}

# Date range requirements
start_date = "2021-06-15"
end_date = "2026-02-15"

hourly_vars = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall", "weather_code", "surface_pressure",
    "et0_fao_evapotranspiration", "vapor_pressure_deficit", "wind_speed_10m",
    "wind_direction_10m", "shortwave_radiation", "direct_radiation"
]

dfs = []

for region_name, coords in regions.items():
    print(f"Downloading from scratch: {region_name} ({start_date} to {end_date})...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": coords["lat"],
        "longitude": coords["lon"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_vars),
        "timezone": "Asia/Kolkata"
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        df_hourly = pd.DataFrame(data.get("hourly", {}))
        
        # Filter for 6-hourly intervals (every 6th hour)
        df_region_6h = df_hourly.iloc[::6].copy()
        
        df_region_6h["region"] = region_name
        df_region_6h["request_date"] = "range_june_2021"
        dfs.append(df_region_6h)
    else:
        print(f"Error for {region_name}: {response.status_code}")

if dfs:
    df_6h = pd.concat(dfs, ignore_index=True)
    output_path = "data/open_meteo_sample.csv"
    
    # Write from scratch (overwrite)
    df_6h.to_csv(output_path, index=False)
    print(f"Fresh data saved to {output_path} ({len(df_6h)} rows).")
else:
    print("No data collected.")

Fresh data saved to data/open_meteo_sample.csv (13656 rows).


## Column Analysis

Analyze the available columns directly from the df_6h dataframe.

In [6]:
if 'df_6h' in locals():
    print("### Column Summary")
    print(df_6h.columns.tolist())
    
    print("\n### Data Head")
    display(df_6h.head())
    
    print("\n### Missing Values per Region/Date")
    analysis = df_6h.groupby(['region', 'request_date']).count()
    display(analysis)
    
    print("\n### Available Variables (excluding metadata columns)")
    vars_available = [c for c in df_6h.columns if c not in ['time', 'region', 'request_date']]
    print(vars_available)

### Column Summary
['time', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'weather_code', 'surface_pressure', 'et0_fao_evapotranspiration', 'vapor_pressure_deficit', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation', 'direct_radiation', 'region', 'request_date']

### Data Head


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,surface_pressure,et0_fao_evapotranspiration,vapor_pressure_deficit,wind_speed_10m,wind_direction_10m,shortwave_radiation,direct_radiation,region,request_date
0,2021-06-15T00:00,28.1,70,22.1,31.2,0.1,0.1,0.0,51,997.8,0.06,1.14,13.7,235,0.0,0.0,Eluru,range_june_2021
1,2021-06-15T06:00,28.4,70,22.5,31.6,0.0,0.0,0.0,3,998.4,0.09,1.14,13.8,261,30.0,4.0,Eluru,range_june_2021
2,2021-06-15T12:00,33.2,55,22.9,36.1,0.0,0.0,0.0,3,997.9,0.48,2.29,16.6,268,512.0,150.0,Eluru,range_june_2021
3,2021-06-15T18:00,30.0,67,23.1,35.1,0.0,0.0,0.0,3,997.2,0.05,1.41,3.6,315,15.0,0.0,Eluru,range_june_2021
4,2021-06-16T00:00,28.9,69,22.8,31.9,0.0,0.0,0.0,3,999.4,0.08,1.22,16.1,243,0.0,0.0,Eluru,range_june_2021



### Missing Values per Region/Date


,,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,surface_pressure,et0_fao_evapotranspiration,vapor_pressure_deficit,wind_speed_10m,wind_direction_10m,shortwave_radiation,direct_radiation
region,request_date,,,,,,,,,,,,,,,,
Eluru,range_june_2021,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828
Nellore,range_june_2021,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828,6828



### Available Variables (excluding metadata columns)
['temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'weather_code', 'surface_pressure', 'et0_fao_evapotranspiration', 'vapor_pressure_deficit', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation', 'direct_radiation']


In [ ]:
if "df_6h" not in locals():
    raise NameError("df_6h is not available. Run the earlier cell that builds it first.")

required_columns = [
    "time",
    "region",
    "request_date",
    "temperature_2m",
    "precipitation",
    "vapor_pressure_deficit",
    "wind_speed_10m",
    "shortwave_radiation",
]
missing_columns = [column for column in required_columns if column not in df_6h.columns]
if missing_columns:
    raise KeyError(f"df_6h is missing required columns: {missing_columns}")

df_6h_features_source = df_6h.copy()
df_6h_features_source["time"] = pd.to_datetime(df_6h_features_source["time"])
df_6h_features_source = df_6h_features_source.sort_values(["region", "time"])

feature_frames = []

for region_name, group in df_6h_features_source.groupby("region", sort=False):
    group = group.sort_values("time").set_index("time")

    # Each window is based on the previous 6-hourly observations only.
    cumulative_shortwave_24h = group["shortwave_radiation"].rolling("24h", closed="left", min_periods=4).sum()
    cumulative_shortwave_48h = group["shortwave_radiation"].rolling("48h", closed="left", min_periods=8).sum()
    max_temperature_24h = group["temperature_2m"].rolling("24h", closed="left", min_periods=4).max()
    min_wind_speed_overnight = group["wind_speed_10m"].rolling("12h", closed="left", min_periods=2).min()
    total_precipitation_48h = group["precipitation"].rolling("48h", closed="left", min_periods=8).sum()
    mean_vpd_24h = group["vapor_pressure_deficit"].rolling("24h", closed="left", min_periods=4).mean()

    six_am = group[group.index.hour == 6].copy()
    six_am["cumulative_shortwave_radiation_prior_24h"] = cumulative_shortwave_24h.reindex(six_am.index).to_numpy()
    six_am["cumulative_shortwave_radiation_prior_48h"] = cumulative_shortwave_48h.reindex(six_am.index).to_numpy()
    six_am["max_temperature_prior_24h"] = max_temperature_24h.reindex(six_am.index).to_numpy()
    six_am["min_wind_speed_overnight"] = min_wind_speed_overnight.reindex(six_am.index).to_numpy()
    six_am["total_precipitation_prior_48h"] = total_precipitation_48h.reindex(six_am.index).to_numpy()
    six_am["mean_vpd_prior_24h"] = mean_vpd_24h.reindex(six_am.index).to_numpy()

    six_am = six_am.reset_index()
    feature_frames.append(
        six_am[
            [
                "time",
                "region",
                "request_date",
                "cumulative_shortwave_radiation_prior_24h",
                "cumulative_shortwave_radiation_prior_48h",
                "max_temperature_prior_24h",
                "min_wind_speed_overnight",
                "total_precipitation_prior_48h",
                "mean_vpd_prior_24h",
            ]
        ]
    )

df_6am_features = pd.concat(feature_frames, ignore_index=True)

feature_columns = [
    "cumulative_shortwave_radiation_prior_24h",
    "cumulative_shortwave_radiation_prior_48h",
    "max_temperature_prior_24h",
    "min_wind_speed_overnight",
    "total_precipitation_prior_48h",
    "mean_vpd_prior_24h",
]

df_6am_features = (
    df_6am_features.sort_values(["region", "time"])
    .dropna(subset=feature_columns)
    .reset_index(drop=True)
)

display(df_6am_features.head())
df_6am_features